In [0]:
!pip install  -U -r requirements.txt "unstructured[local-inference]"

# Hands on Data Preparatin for RAG

**Objective:** The aim of this notebook is to prepare all the data that will be used in the RAG application.   

First, we will apply a very simple text extraction and break it in chuncks. Then we will put that in a vector database applining the databricks fundation embedding model as embedding model. 

In [0]:
import io
import os
import pandas as pd
import json
import fitz

from transformers import AutoTokenizer
from huggingface_hub import login
from llama_index.core.langchain_helpers.text_splitter import SentenceSplitter
from llama_index.core import Document, set_global_tokenizer
from typing import Iterator
from pyspark.sql.functions import udf, pandas_udf, col, length, explode, collect_list
from unstructured.partition.auto import partition
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader
from mlflow.deployments import get_deploy_client

client = get_deploy_client("databricks")

login(token = json.load(open("../config/config.json"))["HF_PAT"])

# set Llama-3.1-8B-Instruct as tokenizer
set_global_tokenizer(
    AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
)

In [0]:
# Defining some variables
articles_path = "/Volumes/analytics/bronze/volume_da/GenAIStudy/"
default_catalog = "analytics"
db_name = "bronze"
table_name = f"pdf_raw_text"

# reading pdf files
df = (spark.read.format("binaryFile")
      .option("recursiveFileLookup", "true")
      .load(articles_path))

# save data to a deltatable
df.write.mode("overwrite").saveAsTable(f"{default_catalog}.{db_name}.{table_name}")

In [0]:
def extract_doc_text(b):
    
    text = ""
    try:
        pdf = fitz.open(stream = b)
        for page in pdf.pages():
            text += page.get_text()

        return text        
        
    except Exception as e:
        print(f"Erro ao carregar arquivo {b}: {e}")

In [0]:
@pandas_udf("array<string>")
def read_as_chuncks(batch_iter: Iterator[pd.Series]) -> Iterator[pd.Series]:
         
    splitter = SentenceSplitter(chunk_size = 500, chunk_overlap = 50)
    def extract_and_split(b):
        txt = extract_doc_text(b)
        nodes = splitter.get_nodes_from_documents([Document(text = txt)])
        print(nodes)
        return [n.text for n in nodes]
    
    for batch in batch_iter:
        yield batch.apply(extract_and_split)

In [0]:
df_chuncks = (df.withColumn("content", explode(read_as_chuncks("content")))
              .selectExpr("path as pdf_name", "content"))

df_chuncks.display()             


In [0]:
@pandas_udf("array<float>")
def get_embedding(content: pd.Series) -> pd.Series:

    
    def get_embeddings(batch):
        response = client.predict(endpoint = "databricks-bge-large-en", inputs = {"input": batch})
        return [r["embedding"] for r in response["data"]]
    

    max_batch_size = 150
    batches = [content.iloc[i:i+max_batch_size] for i in range(0, len(content), max_batch_size)]


    all_embeddings = []
    for batch in batches:
        all_embeddings.extend(get_embeddings(batch.tolist()))
    
    return pd.Series(all_embeddings)   


In [0]:
df_chuncks =  (df_chuncks.withColumn("embeddings", get_embedding("content"))
              .selectExpr("pdf_name", "content", "embeddings")
)

df_chuncks.display()

In [0]:
%sql
CREATE TABLE IF NOT EXISTS  pdf_text_embeddings (
  id BIGINT GENERATED ALWAYS AS IDENTITY,
  pdf_name STRING,
  content STRING,
  embedding ARRAY<FLOAT>
) TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
embedding_tabel_name = "pdf_text_embeddings"
df_chuncks.write.mode("overwrite").saveAsTable(embedding_tabel_name)